# ARC-AGI-2 as region objects

Block composition described 5.1% of the ARC-AGI-2 training split and none of the
evaluation split. **Same-shape tasks** — where the output has the same dimensions
as the input — are 68.0% of training and 67.5% of evaluation, and block
decomposition says nothing about them: their block layout is 1×1, which asserts
nothing at all.

This notebook builds the construction that reaches that population, through the
same four stages:

```text
grid cell observations
    → background and foreground states
    → connected-component identities
    → one row per region object
    → computational queries
```

Unlike block composition, this construction describes **one grid**. It needs no
corresponding output, no task, and no assumed relationship between grids — so it
applies to every grid in the dataset rather than to the subset some hypothesis
happens to fit.

In [ ]:
import pandas as pd

import featuregraph as fg
from featuregraph.behaviors.regions import RegionObjects, edit_alignment

pd.set_option("display.width", 170)

GRID_GROUP = ["task_id", "pair_type", "pair_index", "grid_role"]


def region_objects(task_id, split="training", **construction):
    """Construct region objects for every grid of one ARC-AGI-2 task."""
    observations = fg.datasets.arc_agi(task_id, split=split)
    builder = RegionObjects(
        signals="color",
        group=GRID_GROUP,
        **construction,
    )
    return builder, builder.summarize(builder.fit_transform(observations))

## 1. Region objects

One row per connected region, with the geometry of each made explicit.

In [ ]:
builder, objects = region_objects("1acc24af")

objects.to_pandas()[
    [
        "pair_index",
        "grid_role",
        "region_id",
        "color",
        "is_background",
        "size",
        "height",
        "width",
        "fill_ratio",
        "hole_count",
        "touches_border",
    ]
].head(10)

### The properties support ordinary queries

`duration >= 100` has no meaning against an undifferentiated respiratory trace, and
`hole_count >= 1` has no meaning against an undifferentiated grid. The construction
supplies the identity and boundaries that make the predicate computable.

In [ ]:
(
    objects.query()
    .where(is_background=False, grid_role="input")
    .order_by("size", ascending=False)
    .select("pair_index", "color", "size", "fill_ratio", "hole_count")
    .limit(5)
    .collect()
)

## 2. What holds a region together is declared, not assumed

Connectivity, the background colour, whether a region is a uniform-colour component
or a non-background component, and whether background cells form objects of their
own are all construction parameters — in the same sense that `Oscillation` declares
its smoothing window and difference lag.

Different declarations produce genuinely different objects from identical
observations, so the choice is recorded on the result rather than buried.

In [ ]:
comparison = []
for definition in ("uniform_color", "foreground_mask"):
    for connectivity in (4, 8):
        _, variant = region_objects(
            "1acc24af",
            definition=definition,
            connectivity=connectivity,
            include_background=False,
        )
        comparison.append(
            {
                "definition": definition,
                "connectivity": connectivity,
                "regions": variant.count,
                "median_size": variant.to_pandas()["size"].median(),
            }
        )

pd.DataFrame(comparison)

In [ ]:
objects.construction

## 3. Do the edits respect these boundaries?

This is the question that decides whether regions are the right object boundary.
It is **separate** from whether the transformation can be predicted.

`edit_alignment` groups the cells that differ between an input and its output into
connected edit components, then checks each against the partition the input grid
produces. A component inside one region is aligned; one spanning several is not.

In [ ]:
observations = fg.datasets.arc_agi("1acc24af", split="training")
edits = edit_alignment(observations)

edits.head(10)

In [ ]:
print(f"{edits['is_aligned'].mean():.1%} of edits lie inside a single input region")

Not every task behaves this way. Task `1c0d0a4b` draws across region boundaries,
and the same measurement reports that plainly rather than failing:

In [ ]:
crossing = edit_alignment(fg.datasets.arc_agi("1c0d0a4b", split="training"))

crossing.groupby("regions_spanned").size().rename("edit_components")

## 4. The measurement across the same-shape population

`experiments/arc/region_probe.py` runs this construction over all 761 same-shape
ARC-AGI-2 tasks. It uses the same `RegionObjects` and `edit_alignment` shown above,
so the recorded artifact and the library cannot drift apart.

In [ ]:
probe = pd.read_csv("../artifacts/arc/region_probe.csv")

summary = []
for (split, connectivity), part in probe.groupby(["split", "connectivity"]):
    components = part["change_components"].sum()
    summary.append(
        {
            "split": split,
            "connectivity": connectivity,
            "tasks": len(part),
            "edit_components": int(components),
            "edits_region_aligned": part["aligned_components"].sum() / components,
            "output_colour_predictable": int(
                part[
                    [
                        "colour_determines_output",
                        "size_determines_output",
                        "size_rank_determines_output",
                        "bbox_determines_output",
                    ]
                ]
                .any(axis=1)
                .sum()
            ),
        }
    )

pd.DataFrame(summary)

Two results, and they must not be run together.

**Regions are the right object boundary.** 94.2% of training edit components and
94.0% of evaluation edit components lie entirely inside a single input region.
Under 8-connectivity, 91.7% and 90.5% — not an artefact of that choice.

**Region edits are not predictable from simple region properties.** Output colour
is determined by input colour, size, size rank, or bounding box for 10 training
tasks and 0 evaluation tasks. The obvious recolour vocabulary does not work.

The first result is what a representational claim needs. The second is a fact about
a hypothesis class, reported rather than chased.

### Why this construction and not more block operators

| | training | evaluation |
| --- | ---: | ---: |
| block composition, non-trivial tiling | 5.1% | 0.0% |
| region construction, edits aligned | 94.2% | 94.0% |

Block composition collapsed between splits. The region construction does not —
which is the property a cross-split representational claim actually requires.

See [`artifacts/arc/region_objects_scope.md`](../artifacts/arc/region_objects_scope.md)
for what was measured before building this, and what is deliberately out of scope:
an operator vocabulary for region edits. Coverage is the measurement, not the
target.